# Does this agree with the people who were there?

*Question → Intuition → Math → Code → Assumptions → How it breaks*

## 1. Question

Every check in this book so far has been internal. The gate is consistent with
the requirement list, the requirement list with the definition, the intervals
with the data. All of that could be internally perfect while measuring the wrong
thing entirely.

So: does this ranking agree with the people who watched the football and voted
at the time — and where it disagrees, can it say why?

## 2. Intuition

Ballon d'Or voting is the one outside opinion available, and it needs no new
scraping: it is already in Wikidata.

It is **not ground truth**, and the temptation to treat it as such should be
resisted in both directions. It rewards trophies won by teammates. It is swayed
by World Cups this project cannot see. It leans toward forwards, and toward
whoever won the Champions League that year. A ranking that reproduced it
perfectly would have reproduced its biases too.

That is exactly why it is worth comparing against rather than fitting to.
Agreement is evidence the method measures something real. **Disagreement is the
more interesting output**, because each disagreement has a nameable cause — and
a method that cannot name it is not defensible.

## 3. Math

There is deliberately no single statistic here.

The obvious move is a rank correlation — Spearman's $\rho$ between award count
and our rank. It would be a mistake. Only eighteen players won anything in the
window, so $\rho$ would be computed on a tiny, extremely selected subset, and
would compress the one thing worth seeing into a single number.

Three counts instead, kept apart on purpose:

1. winners **present** in our data at all,
2. winners who **clear the gate**,
3. winners the gate **rejects**, with the requirement responsible.

Keeping (1) separate from (3) matters. "We never saw this player" is a data
failure and "we ranked this player badly" is a method disagreement, and adding
them together would hide whichever is ours.

In [ ]:
import warnings

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
matplotlib.rcParams["figure.figsize"] = (10, 5.5)

SAMPLE = "../data/sample"
ranking = pd.read_parquet(f"{SAMPLE}/ranking.parquet")
winners = pd.read_csv(f"{SAMPLE}/award_winners.csv")
agreement = pd.read_csv(f"{SAMPLE}/awards_agreement.csv")

In [ ]:
agreement.style.hide(axis="index")

Ten of sixteen located winners clear all eleven requirements, and the
median winner ranks 61st of 5,508 — the top 1.1%. For a definition written
without reference to any award, that is substantial agreement.

Now the part worth reading.

In [ ]:
placed = winners[winners["rank"].notna()].copy()
placed["verdict"] = np.where(
    placed["qualified"].fillna(False).astype(bool),
    "qualified",
    "failed: " + placed["failed"].fillna("-"),
)
(
    placed.sort_values("rank")[["player", "honours", "award_years", "rank", "verdict"]].style.hide(
        axis="index"
    )
)

## 4. Where it disagrees, and why

Read the bottom of that table rather than the top.

**Cannavaro, 2,707th.** He won the 2006 Ballon d'Or as a centre-back and this
definition ranks him below two and a half thousand players, failing eight of
eleven requirements. This is not a bug to be fixed — it is the
attacking-contribution limitation stated as a number instead of a caveat. FBref
records no per-player defensive action before 2017-18, so two thirds of the
window has nothing to measure a defender with. Van Dijk, 2019, fails on creation
for the same reason.

**Three winners fail on discipline alone.** Nedvěd, Figo and Rodri clear ten
requirements and are stopped by one. Whether that is right is a live question:
the definition weights being booked like being unavailable, and nobody has
argued that those deserve equal footing.

In [ ]:
blame = (
    placed[~placed["qualified"].fillna(False).astype(bool)]["failed"]
    .str.split(", ")
    .explode()
    .value_counts()
)
print("requirements responsible for rejecting an award winner:\n")
print(blame.to_string())
print("\nDiscipline appears more than anything else, on players nobody would")
print("call undisciplined enough to disqualify. That is a finding about the")
print("definition, not about the players.")

In [ ]:
fig, ax = plt.subplots()
ax.hist(np.log10(ranking.index + 1), bins=60, color="#DDD", edgecolor="none")
for row in placed.nlargest(4, "rank").itertuples():
    ax.axvline(np.log10(row.rank), color="#D62728", linewidth=1.5)
    ax.text(
        np.log10(row.rank),
        ax.get_ylim()[1] * 0.9,
        f" {row.player}",
        rotation=90,
        va="top",
        fontsize=9,
        color="#D62728",
    )
for row in placed.nsmallest(3, "rank").itertuples():
    ax.axvline(np.log10(row.rank), color="#2CA02C", linewidth=1.5)
ax.set_xlabel("our rank (log scale)")
ax.set_ylabel("players")
ax.set_title("Where award winners land in our ranking")
plt.show()

## 5. Assumptions

1. **Award voting tracks playing quality.** Loosely, and with the biases named
   above. It is a noisy second opinion, not a criterion.
2. **The window is comparable.** Awards are given for a calendar year and this
   project measures seasons, so a 2010 award spans two of our seasons. At this
   resolution the mismatch does not matter; for a season-by-season comparison it
   would.
3. **Identity resolution found them.** A winner we cannot match looks identical
   to a winner we ranked badly, which is why the two are reported separately.

## 6. How it breaks

**This check found two bugs in itself before it found anything about football**,
which is worth recording rather than quietly fixing.

The first run reported Cristiano Ronaldo — seven honours — as a player the data
had never seen. He was not missing from the ranking; he sits second. The Wikidata
fetch had silently lost the 1985 birth cohort, and 1985 is when both he and
Modrić were born.

The same run counted Birgit Prinz, Carli Lloyd and Aitana Bonmatí as winners we
had failed to find. Their absence is correct — women's football is a declared
non-goal — but the query had not said so, and three correct absences were being
reported as our failures.

**The deeper limit: agreement here cannot validate the parts that matter most.**
Awards go overwhelmingly to forwards, so an attacking-contribution rating and a
forward-biased vote will agree for reasons that have nothing to do with either
being right. The agreement is real evidence, but it is evidence about the
population the two measures share — and it is silent on defenders, which is
precisely where this project is weakest.